In [1]:
# !unzip Hollywood/small_Annotations.zip -d Hollywood/
import torch
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
import os
import torch
import xml.etree.ElementTree as ET
from PIL import Image
class VOCDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, ann_dir,transforms=None):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.transforms = transforms


        self.imgs = sorted([
            f for f in os.listdir(img_dir)
            if os.path.isfile(os.path.join(img_dir, f))
               and f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

    def parse_xml(self, xml_path, img_width, img_height):
        tree = ET.parse(xml_path)
        root = tree.getroot()

        boxes = []
        labels = []

        for obj in root.findall("object"):
            labels.append(1)

            box = obj.find("bndbox")
            xmin = float(box.find("xmin").text)
            ymin = float(box.find("ymin").text)
            xmax = float(box.find("xmax").text)
            ymax = float(box.find("ymax").text)

            # clamp boxes
            xmin = max(0, min(xmin, img_width  - 1))
            xmax = max(0, min(xmax, img_width  - 1))
            ymin = max(0, min(ymin, img_height - 1))
            ymax = max(0, min(ymax, img_height - 1))

            boxes.append([xmin, ymin, xmax, ymax])

        return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)


    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)

        base = os.path.splitext(img_name)[0]
        ann_path = os.path.join(self.ann_dir, base + ".xml")

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes, labels = self.parse_xml(ann_path, w, h)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]),
            "iscrowd": torch.zeros((len(labels),), dtype=torch.int64)
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.imgs)




In [4]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
transforms_tool=transforms.Compose([transforms.ToTensor(),transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),])
dataset=VOCDataset("./Hollywood/small_JPEGImages","./Hollywood/small_Annotations",transforms=transforms_tool)
train_size=int(len(dataset)*0.9)
test_size=len(dataset)-train_size
train_dataset,testdataset=torch.utils.data.random_split(dataset,[train_size,test_size])
train_loader=DataLoader(train_dataset,shuffle=True,batch_size=8,collate_fn=lambda x: tuple(zip(*x)),num_workers=8,pin_memory=True)


In [2]:
import torchvision
import torch.nn as nn
from torchvision.models.detection import retinanet_resnet50_fpn

model = retinanet_resnet50_fpn(pretrained=True)

num_classes = 2  # 背景 + 头部 (注意：RetinaNet会自动处理背景类，所以这里是1)

# 3. 获取分类头中最后一个卷积层的输入通道数
in_channels = model.head.classification_head.cls_logits.in_channels

# 4. 计算新的输出通道数
#    RetinaNet的每个锚点都需要预测所有类别的得分
#    输出通道数 = 锚点数量 * 类别数
#    锚点数量通常是9，这个值可以从模型的anchor_generator中获取
num_anchors = model.anchor_generator.num_anchors_per_location()[0]
out_channels = num_anchors * num_classes

# 5. 创建一个新的卷积层来替换旧的
#    注意：权重会被随机初始化，这是正常的，因为我们要在新任务上微调
new_cls_logits = nn.Conv2d(
    in_channels,
    out_channels,
    kernel_size=3,
    stride=1,
    padding=1
)
# 6. 替换模型中的旧层
model.head.classification_head.cls_logits = new_cls_logits

# 7. (重要) 更新模型的num_classes属性
#    这一步是为了让模型内部的某些逻辑（比如损失函数计算）知道新的类别数
model.head.classification_head.num_classes = num_classes
# 回归头的输入通道数与分类头相同
in_channels_reg = in_channels 
# 回归头的输出通道数 = 每个锚点预测的坐标数 * 锚点数量
# 每个锚点预测 4 个值 (dx, dy, dw, dh)
out_channels_reg = num_anchors * 4
model.head.regression_head.bbox_reg = nn.Conv2d(
    in_channels_reg,
    out_channels_reg,
    kernel_size=3,
    stride=1,
    padding=1
)




D:\Anaconda\envs\ai4beg\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\Anaconda\envs\ai4beg\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=RetinaNet_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=RetinaNet_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/retinanet_resnet50_fpn_coco-eeacb38b.pth" to C:\Users\huawei/.cache\torch\hub\checkpoints\retinanet_resnet50_fpn_coco-eeacb38b.pth
100%|██████████| 130M/130M [00:16<00:00, 8.13MB/s] 


In [8]:
model.train()
print(model.head)


RetinaNetHead(
  (classification_head): RetinaNetClassificationHead(
    (conv): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
      )
      (1): Conv2dNormActivation(
        (0): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
      )
      (2): Conv2dNormActivation(
        (0): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
      )
      (3): Conv2dNormActivation(
        (0): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
      )
    )
    (cls_logits): Conv2d(256, 18, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
  (regression_head): RetinaNetRegressionHead(
    (conv): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1):

In [ ]:
print(model.head.classification_head.num_classes)
print(model.head.classification_head.cls_logits.out_channels)

In [7]:
def train(model,train_loader,epochs,device):
    model=model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    # optimizer = torch.optim.SGD(
    #     model.parameters(), 
    #     lr=1e-3,  # 学习率可以从 1e-3 开始尝试
    #     momentum=0.9,  # 动量有助于稳定训练
    #     weight_decay=1e-4  # 权重衰减，防止过拟合
    # )
    for epoch in range(epochs):
        model.train()
        model.head.score_thresh = 0.0
        for i,(images,targets) in enumerate(train_loader):
            
            images=[img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            loss_dict=model(images,targets)
            if(i%50==0):
                print(loss_dict)
            loss = sum(loss_dict.values())
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            if i%10==0:
                print(f"i: {i} Loss: {loss}")
        print(f"Epoch {epoch}: loss={loss.item():.4f}")

In [ ]:
torch.backends.cudnn.benchmark = True
train(model,train_loader,10,device)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
os
for i,img_name in enumerate(os.listdir("./Hollywood/small_JPEGImages")):
    if(i%100==0):
        img_path=os.path.join("./Hollywood/small_JPEGImages",img_name)
        image = Image.open(img_path).convert("RGB")
        original_image = image.copy() # 保存原始图片用于后续绘图

        preprocess = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        image_tensor=preprocess(image).unsqueeze(0).to(device)

        model.eval()
        with torch.no_grad(): # 关闭梯度计算
            predictions = model(image_tensor)
        print(predictions)
        predictions = [{k: v.to('cpu') for k, v in t.items()} for t in predictions]
        threshold = 0.3
        pred_boxes = predictions[0]['boxes'][predictions[0]['scores'] > threshold]
        pred_labels = predictions[0]['labels'][predictions[0]['scores'] > threshold]
        pred_scores = predictions[0]['scores'][predictions[0]['scores'] > threshold]

        # 类别名称映射（根据你的训练数据）
        label_map = {0: 'background', 1: 'person'} # 示例

        # 绘制结果
        fig, ax = plt.subplots(1, figsize=(12, 9))
        ax.imshow(original_image)

        for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
            if label.item() == 1:
                x1, y1, x2, y2 = box
                width, height = x2 - x1, y2 - y1
                rect = patches.Rectangle((x1, y1), width, height, linewidth=2, edgecolor='r', facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1 - 10, f"{label_map[label.item()]}: {score:.2f}", color='r', fontsize=12)

        plt.axis('off')
        plt.show()
    if i==3000:
        break